In [17]:
## add_circuit

from ortools.sat.python import cp_model


dgraph = {
    (0, 1): 10, (1, 0): 12,
    (0, 2): 15, (2, 0): 9,
    (0, 3): 20, (3, 0): 8,
    (1, 2): 7,  (2, 1): 14,
    (1, 3): 11, (3, 1): 6,
    (2, 3): 5,  (3, 2): 13,
}
# Initialize CP-SAT model
model = cp_model.CpModel()

# Boolean variables for each edge
edge_vars = {(u, v): model.new_bool_var(f"e_{u}_{v}") for (u, v) in dgraph.keys()}

print(edge_vars)

# Circuit constraint for a single tour
model.add_circuit([(u, v, var) for (u, v), var in edge_vars.items()])

# Objective function to minimize total cost
model.minimize(sum(dgraph[(u, v)] * x for (u, v), x in edge_vars.items()))

# Solve model
solver = cp_model.CpSolver()
status = solver.solve(model)
if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    tour = [(u, v) for (u, v), x in edge_vars.items() if solver.value(x)]
    print("Tour:", tour)

{(0, 1): e_0_1(0..1), (1, 0): e_1_0(0..1), (0, 2): e_0_2(0..1), (2, 0): e_2_0(0..1), (0, 3): e_0_3(0..1), (3, 0): e_3_0(0..1), (1, 2): e_1_2(0..1), (2, 1): e_2_1(0..1), (1, 3): e_1_3(0..1), (3, 1): e_3_1(0..1), (2, 3): e_2_3(0..1), (3, 2): e_3_2(0..1)}
Tour: [(0, 1), (3, 0), (1, 2), (2, 3)]


In [27]:
## Creative Usage Off Add Circuit

from ortools.sat.python import cp_model

# Define a weighted, directed graph with edge costs
dgraph = {
    (0, 1): 10, (1, 0): 12,
    (0, 2): 15, (2, 0): 9,
    (0, 3): 20, (3, 0): 8,
    (1, 2): 7,  (2, 1): 14,
    (1, 3): 11, (3, 1): 6,
    (2, 3): 5,  (3, 2): 13,
}
source_vertex = 0
target_vertex = 3

# Add zero-cost loops for vertices not being the source or target
for v in [1, 2]:
    dgraph[(v, v)] = 0

# Initialize CP-SAT model and variables
model = cp_model.CpModel()
edge_vars = {(u, v): model.new_bool_var(f"e_{u}_{v}") for (u, v) in dgraph}

# Define the circuit including a pseudo-edge from target to source
circuit = [(u, v, var) for (u, v), var in edge_vars.items()] + [
    (target_vertex, source_vertex, 1)
]
print(circuit)
model.add_circuit(circuit)

# Minimize total cost
model.minimize(sum(dgraph[(u, v)] * x for (u, v), x in edge_vars.items()))

# Solve and extract the path
solver = cp_model.CpSolver()
status = solver.solve(model)

for (u, v), x in edge_vars.items():
    print(f"e_{u}_{v} = {solver.value(x)}")
if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    path = [(u, v) for (u, v), x in edge_vars.items() if solver.value(x) and u != v]
    print("Path:", path)



[(0, 1, e_0_1(0..1)), (1, 0, e_1_0(0..1)), (0, 2, e_0_2(0..1)), (2, 0, e_2_0(0..1)), (0, 3, e_0_3(0..1)), (3, 0, e_3_0(0..1)), (1, 2, e_1_2(0..1)), (2, 1, e_2_1(0..1)), (1, 3, e_1_3(0..1)), (3, 1, e_3_1(0..1)), (2, 3, e_2_3(0..1)), (3, 2, e_3_2(0..1)), (1, 1, e_1_1(0..1)), (2, 2, e_2_2(0..1)), (3, 0, 1)]
e_0_1 = 0
e_1_0 = 0
e_0_2 = 0
e_2_0 = 0
e_0_3 = 1
e_3_0 = 0
e_1_2 = 0
e_2_1 = 0
e_1_3 = 0
e_3_1 = 0
e_2_3 = 0
e_3_2 = 0
e_1_1 = 1
e_2_2 = 1
Path: [(0, 3)]


In [11]:
from typing import Hashable
import networkx as nx
from ortools.sat.python import cp_model


class CvrpMultiCircuit:
    """CVRP via CP-SAT multi-circuit constraint."""

    def __init__(
        self,
        graph: nx.Graph,
        depot: Hashable, ##Where all trips start
        capacity: int,
        demand_label: str = "demand",
        model: cp_model.CpModel | None = None,
    ):
        self.graph, self.depot = graph, depot
        self.model = model or cp_model.CpModel()
        self.capacity = capacity
        self.demand_label = demand_label

        # Vertex list with depot first
        self.vertices = [depot] + [v for v in graph.nodes() if v != depot]
        print("Vertices", self.vertices)
        self.index = {v: i for i, v in enumerate(self.vertices)}
        print("index", self.index)
        print("Graph edges", graph.edges)
        # Boolean arc variables for both directions
        self.arc_vars = {
            (i, j): self.model.new_bool_var(f"arc_{i}_{j}")
            for u, v in graph.edges
            for i, j in ((self.index[u], self.index[v]), (self.index[v], self.index[u]))
        }
        arcs = [(i, j, var) for (i, j), var in self.arc_vars.items()]
        # Multi-circuit constraint
        self.model.add_multiple_circuit(arcs)

        # Capacity variables and constraints
        self.cap_vars = [
            self.model.new_int_var(0, capacity, f"cap_{i}")
            for i in range(len(self.vertices))
        ]
        
        print("cap_vars",self.cap_vars)
        for i, j, var in arcs:
            if j == 0:
                continue
            demand = graph.nodes[self.vertices[j]].get(demand_label, 0)
            ct = self.model.add(
                self.cap_vars[j] >= self.cap_vars[i] + demand ## running total at j must be at least the running total at i + the demand at j.
            ).only_enforce_if(var) #Only applies if  route goes from i -j
            # THis constraint forces the model to not go over capacity because cap_vars cannot be greater greater than capacity(0..6).

    def is_arc_used(self, u, v) -> cp_model.BoolVarT:
        return self.arc_vars[(self.index[u], self.index[v])]

    def weight(self, label: str = "weight") -> cp_model.LinearExprT:
        print(self.graph[self.vertices[0]][self.vertices[1]][label])
        return sum(
            var * self.graph[self.vertices[i]][self.vertices[j]][label]
            for (i, j), var in self.arc_vars.items()
        )

    def minimize_weight(self, label: str = "weight"):
        self.model.minimize(self.weight(label=label))

    def extract_tours(self, solver: cp_model.CpSolver) -> list[list]:
        # Build directed graph of selected arcs
        dg = nx.DiGraph(
            [
                (self.vertices[i], self.vertices[j])
                for (i, j), var in self.arc_vars.items()
                if solver.value(var)
            ]
        )

        # Eulerian circuit and split at depot
        euler = nx.eulerian_circuit(dg, source=self.depot)
        tours, curr = [], [self.depot]
        for u, v in euler:
            curr.append(v)
            if v == self.depot:
                tours.append(curr)
                curr = [self.depot]
        if len(curr) > 1:
            tours.append(curr)
        return tours
    
    

G = nx.Graph()
G.add_node("depot")
G.add_node("A", demand=3) ## Demand = how much a customer needs,(Can't exceed capacity)
G.add_node("B", demand=5)
G.add_node("C", demand=2)

G.add_edge("depot", "A", weight=4) ## WEight cost os using road (Objective is to minimize toal)
G.add_edge("depot", "B", weight=6)
G.add_edge("depot", "C", weight=5)
G.add_edge("A", "B", weight=2)
G.add_edge("B", "C", weight=3)
G.add_edge("A", "C", weight=7)
print(G.nodes)

cvrp = CvrpMultiCircuit(G, depot="depot", capacity=6)
cvrp.minimize_weight()

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 10
status = solver.solve(cvrp.model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("Status:", solver.status_name(status))
    print("Total weight:", solver.objective_value)
    print("Tours:", cvrp.extract_tours(solver))
else:
    print("No solution:", solver.status_name(status))


['depot', 'A', 'B', 'C']
Vertices ['depot', 'A', 'B', 'C']
index {'depot': 0, 'A': 1, 'B': 2, 'C': 3}
Graph edges [('depot', 'A'), ('depot', 'B'), ('depot', 'C'), ('A', 'B'), ('A', 'C'), ('B', 'C')]
cap_vars [cap_0(0..6), cap_1(0..6), cap_2(0..6), cap_3(0..6)]
4
Status: OPTIMAL
Total weight: 28.0
Tours: [['depot', 'B', 'depot'], ['depot', 'C', 'A', 'depot']]
